# Practice Lab: Math Foundations 2 - How Models Learn

8 exercises on cross-entropy, gradient descent, learning rate, backpropagation.

In [ ]:
!pip install -q numpy torch matplotlib scikit-learn

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
print('torch:', torch.__version__)

## Exercise 1 - Cross-entropy by hand

In [ ]:
logits = np.array([1.5, 0.5, 2.0])
true_idx = 2

shifted = logits - logits.max()
exps = np.exp(shifted)
probs = exps / exps.sum()
ce_hand = -np.log(probs[true_idx])
print(f'by hand: {ce_hand:.6f}')

ce_pt = F.cross_entropy(torch.tensor(logits).unsqueeze(0).float(), torch.tensor([true_idx]))
print(f'pytorch: {ce_pt.item():.6f}')
assert abs(ce_hand - ce_pt.item()) < 1e-5

## Exercise 2 - Three LR regimes

In [ ]:
def loss(w): return (w - 3) ** 2
def grad(w): return 2 * (w - 3)

def descend(lr, steps=50, w0=0.0):
    w = w0
    for _ in range(steps):
        w = w - lr * grad(w)
        if abs(w) > 1e10: return float('inf'), float('inf')
    return w, loss(w)

for lr in [0.001, 0.1, 1.5]:
    w_final, l_final = descend(lr)
    print(f'LR={lr:.3f}  w_final={w_final:.3e}  loss={l_final:.3e}')

## Exercise 3 - Manual gradient vs autograd

In [ ]:
w = torch.tensor([1.0, 1.0], requires_grad=True)
loss = w[0] ** 2 + 2 * w[1] ** 2
loss.backward()
print(f'autograd grad: {w.grad.tolist()}')
# Expected: [2.0, 4.0]

## Exercise 4 - The zero_grad bug

In [ ]:
torch.manual_seed(0)
x = torch.randn(64, 4)
y = torch.randn(64, 1)

def train(zero_grad):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    losses = []
    for step in range(20):
        if zero_grad: opt.zero_grad()
        pred = model(x)
        l = F.mse_loss(pred, y)
        l.backward()
        opt.step()
        losses.append(l.item())
        if l.item() > 1e10:
            losses.extend([float('inf')] * (20 - step - 1))
            break
    return losses

buggy = train(zero_grad=False)
fixed = train(zero_grad=True)
print('step  buggy        fixed')
for i, (b, f) in enumerate(zip(buggy, fixed)):
    print(f'{i:4d}  {b:.3e}  {f:.3e}')

## Exercise 5 - Cross-entropy vs MSE convergence

In [ ]:
torch.manual_seed(0)
N, D, C = 256, 8, 4
X = torch.randn(N, D)
W_true = torch.randn(D, C)
y = (X @ W_true).argmax(dim=1)
y_onehot = F.one_hot(y, C).float()

def train(loss_fn_kind):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(D, 16), nn.ReLU(), nn.Linear(16, C))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    losses = []
    for step in range(200):
        opt.zero_grad()
        logits = model(X)
        if loss_fn_kind == 'ce':
            l = F.cross_entropy(logits, y)
        else:
            l = F.mse_loss(logits.softmax(dim=1), y_onehot)
        l.backward()
        opt.step()
        losses.append(l.item())
    return losses

ce_curve = train('ce')
mse_curve = train('mse')
print(f'CE  final: {ce_curve[-1]:.4f}, halfway: {ce_curve[100]:.4f}')
print(f'MSE final: {mse_curve[-1]:.4f}, halfway: {mse_curve[100]:.4f}')

## Exercise 6 - Gradient clipping stability

In [ ]:
def train(use_clip):
    torch.manual_seed(0)
    model = nn.Linear(4, 1)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    losses = []
    for step in range(50):
        opt.zero_grad()
        x = torch.randn(32, 4)
        if step % 10 == 9: x = x * 50.0
        y = x.sum(dim=1, keepdim=True) + torch.randn(32, 1) * 0.1
        pred = model(x)
        l = F.mse_loss(pred, y)
        l.backward()
        if use_clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        losses.append(l.item())
    return losses

noclip = train(False)
withclip = train(True)
print(f'without clip: max loss = {max(noclip):.2e}')
print(f'with clip:    max loss = {max(withclip):.2e}')

## Exercise 7 - Sigmoid vanishing gradients

In [ ]:
def build(activation):
    layers = []
    in_dim = 4
    for i in range(10):
        layers.append(nn.Linear(in_dim, 4))
        layers.append(activation())
        in_dim = 4
    layers.append(nn.Linear(4, 1))
    return nn.Sequential(*layers)

def grad_norms(model, x, y):
    pred = model(x)
    l = F.mse_loss(pred, y)
    l.backward()
    return [m.weight.grad.norm().item() for m in model if isinstance(m, nn.Linear)]

torch.manual_seed(0)
x = torch.randn(32, 4)
y = torch.randn(32, 1)
print('Sigmoid grad norms (input -> output):')
print([f'{v:.2e}' for v in grad_norms(build(nn.Sigmoid), x, y)])
print('ReLU grad norms (input -> output):')
print([f'{v:.2e}' for v in grad_norms(build(nn.ReLU), x, y)])

## Exercise 8 - Cosine LR with warmup

In [ ]:
import math

def get_lr(step, total_steps, peak_lr=1e-3, warmup_frac=0.02, min_lr_frac=0.1):
    warmup_steps = int(total_steps * warmup_frac)
    if step < warmup_steps:
        return peak_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cos = 0.5 * (1 + math.cos(math.pi * progress))
    return peak_lr * (min_lr_frac + (1 - min_lr_frac) * cos)

total = 1000
schedule = [get_lr(s, total) for s in range(total)]
print(f'step 0:    LR = {schedule[0]:.2e}')
print(f'step 20:   LR = {schedule[20]:.2e} (end of warmup)')
print(f'step 500:  LR = {schedule[500]:.2e} (midway)')
print(f'step 999:  LR = {schedule[999]:.2e} (10% of peak)')

plt.plot(schedule)
plt.xlabel('step'); plt.ylabel('LR'); plt.title('Cosine schedule with linear warmup')
plt.show()

## Wrap-up

Exercise 4 and Exercise 7 are the two production-training failure modes you are most likely to encounter. Exercise 8 (cosine schedule) is the production standard.